In [0]:
# ============================================
# 补充 1: 窗口函数实战
# ============================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

cur_base = "abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/"

# 1. 读取 FactSales
fact_sales = spark.read.format("delta").load(cur_base + "FactSales/")

# 2. 先聚合：每个客户的总销售额
customer_sales = (fact_sales
    .groupBy("CustomerKey")
    .agg(F.sum("LineTotal").alias("TotalSales"))
)

# 3. 加窗口函数：按销售额排名
window_spec = Window.orderBy(F.desc("TotalSales"))

ranked = (customer_sales
    .withColumn("Rank", F.rank().over(window_spec))
    .withColumn("DenseRank", F.dense_rank().over(window_spec))
    .withColumn("RowNumber", F.row_number().over(window_spec))
)

# 4. 看结果
display(ranked.orderBy("Rank").limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CustomerKey,TotalSales,Rank,DenseRank,RowNumber
335,877107.1923,1,1,1
232,853849.1795,2,2,2
239,841908.7708,3,3,3
720,816755.5763,4,4,4
131,799277.8953,5,5,5
156,787773.0436,6,6,6
218,746317.5293,7,7,7
134,740985.8338,8,8,8
511,730798.7139,9,9,9
163,727272.6494,10,10,10


In [0]:
# 制造并列测试
from pyspark.sql import functions as F
from pyspark.sql.window import Window

test_data = [
    (1, 100.0),
    (2, 100.0),   # 和 Customer 1 并列
    (3, 90.0),
    (4, 80.0),
]

df_test = spark.createDataFrame(test_data, ["CustomerKey", "TotalSales"])

window_spec = Window.orderBy(F.desc("TotalSales"))

df_test.select(
    "CustomerKey", "TotalSales",
    F.rank().over(window_spec).alias("Rank"),
    F.dense_rank().over(window_spec).alias("DenseRank"),
    F.row_number().over(window_spec).alias("RowNumber")
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+----------+----+---------+---------+
|CustomerKey|TotalSales|Rank|DenseRank|RowNumber|
+-----------+----------+----+---------+---------+
|          1|     100.0|   1|        1|        1|
|          2|     100.0|   1|        1|        2|
|          3|      90.0|   3|        2|        3|
|          4|      80.0|   4|        3|        4|
+-----------+----------+----+---------+---------+



In [0]:
# 挑战：每个客户的累计销售额
window_cumulative = (Window
    .partitionBy("CustomerKey")
    .orderBy("DateKey")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

cumulative = (fact_sales
    .withColumn("CumulativeSales", F.sum("LineTotal").over(window_cumulative))
    .select("CustomerKey", "DateKey", "LineTotal", "CumulativeSales")
    .orderBy("CustomerKey", "DateKey")
)

display(cumulative.limit(20))

CustomerKey,DateKey,LineTotal,CumulativeSales
1,20110801,4049.9880,4049.9880
1,20120129,4079.9880,8129.9760
1,20120430,1104.9968,9234.9728
1,20120731,3239.9700,21254.4656
1,20120731,28.8404,16435.9622
1,20120731,1488.5454,18014.4956
1,20120731,1472.2910,23230.3901
1,20120731,182.3520,21616.7936
1,20120731,627.0750,16046.8638
1,20120731,68.3820,15419.7888
